# Create the Customer H2O Environment

The registered model is only one half of the deployment. This notebook generates a separate Azure ML environment from the target-runtime versions recorded during intake. The root `.venv` is not changed and does not need to match the model producer.

## Before you run it

- Complete the package and model-registration notebooks.
- Confirm `H2O_CUSTOMER_MODEL_PATH` still selects the bundle validated and registered in notebooks 01 and 02. See `data/h2o/customer_bundle/README.md` before replacing the demo.
- For a new native binary, MOJO, or runtime, rerun from notebook 01 and assign new immutable model and environment versions.
- Confirm `H2O_CUSTOMER_ENVIRONMENT_NAME` in `workshop/.env`.
- Leave `REGISTER_H2O_ENVIRONMENT=false` on the first pass so you can review the generated Conda file.

We will read the runtime contract, generate the Conda definition, prepare the Azure ML environment, confirm the target workspace, and register only when the safety switch is enabled.

> Native binaries require a matching producer/runtime H2O version. MOJOs may use a separately selected compatible runtime. When golden fixtures are present, notebook 04 verifies that runtime before optional promotion.

**Source:** Adapted from this repository's H2O deployment notebook and the Azure ML environment examples.

In [ ]:
from pathlib import Path
import json
import os

from azure.ai.ml import MLClient
from azure.ai.ml.entities import Environment
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")

load_dotenv(WORKSHOP_ROOT / ".env", override=True)
print(f"Workshop root: {WORKSHOP_ROOT}")

## 1. Read the validated runtime contract

The manifest created during intake is the source of truth for model format, producer version, Python, Java, and selected H2O scoring runtime. We compare the runtime with `.env`, reject mutable versions, and check that the inference server supports the requested Python version.

If an older customer model needs Python below 3.9, stop here. It needs a separately reviewed compatibility image rather than an untested package change.

In [ ]:
model_value = Path(os.environ["H2O_CUSTOMER_MODEL_PATH"])
MODEL_PATH = (
    model_value if model_value.is_absolute() else WORKSHOP_ROOT / model_value
)
BUNDLE_DIR = MODEL_PATH.resolve().parent
manifest_path = BUNDLE_DIR / "model_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

configured_runtime_h2o_version = os.getenv(
    "H2O_CUSTOMER_RUNTIME_VERSION",
    os.getenv("H2O_CUSTOMER_VERSION", ""),
)
if manifest["runtime_h2o_version"] != configured_runtime_h2o_version:
    raise RuntimeError(
        "The customer manifest and configured H2O runtime versions differ"
    )

ENVIRONMENT_NAME = os.environ["H2O_CUSTOMER_ENVIRONMENT_NAME"]
REGISTER = (
    os.getenv("REGISTER_H2O_ENVIRONMENT", "false").lower()
    in {"1", "true", "yes"}
)
PYTHON_VERSION = manifest["python_version"]
JAVA_VERSION = manifest["java_version"]
H2O_PIP_SPEC = (
    manifest.get("h2o_pip_spec")
    or f"h2o=={manifest['runtime_h2o_version']}"
)

python_parts = tuple(int(part) for part in PYTHON_VERSION.split(".")[:2])
if python_parts < (3, 9):
    raise RuntimeError(
        "azureml-inference-server-http==1.4.1 requires Python >=3.9; "
        "this customer runtime needs a separately approved compatibility container"
    )

display(
    {
        "environment": f"{ENVIRONMENT_NAME} (version assigned on registration)",
        "python": PYTHON_VERSION,
        "java": JAVA_VERSION,
        "model_h2o": manifest["h2o_version"],
        "model_format": manifest["model_format"],
        "mojo": manifest.get("mojo_version") or "not applicable",
        "runtime_h2o": manifest["runtime_h2o_version"],
        "h2o_pip_spec": H2O_PIP_SPEC,
    }
)

## 2. Generate and review the Conda file

The Conda file pins Python, OpenJDK, the selected H2O scoring package, and the Azure ML inference dependencies. We write it under `workshop/outputs` so generated runtime files stay separate from source files.

Read the printed YAML before moving on. It should match the customer contract exactly.

In [ ]:
output_dir = WORKSHOP_ROOT / "outputs/generated/h2o_customer/environment"
output_dir.mkdir(parents=True, exist_ok=True)
conda_path = output_dir / "conda.yaml"

conda_content = f"""name: customer-h2o-runtime
channels:
  - conda-forge
dependencies:
  - python={PYTHON_VERSION}
  - openjdk={JAVA_VERSION}
  - pip
  - pip:
      - azureml-inference-server-http==1.4.1
      - {json.dumps(H2O_PIP_SPEC)}
      - numpy==1.26.4
      - pandas==2.2.3
      - mlflow==2.22.1
      - azureml-mlflow==1.60.0.post1
"""

conda_path.write_text(conda_content, encoding="utf-8")

print(f"Conda file written to: {conda_path}")
print(conda_content)

## 3. Prepare the Azure ML environment

The environment combines a Microsoft Azure ML base image with the generated Conda file. The same immutable environment loads the native binary or imports the MOJO for online and offline scoring.

In [ ]:
environment_definition = Environment(
    name=ENVIRONMENT_NAME,
    image="mcr.microsoft.com/azureml/openmpi5.0-ubuntu24.04:latest",
    conda_file=str(conda_path),
    description=(
        "Customer-provided H2O runtime for online and offline scoring"
    ),
    tags={
        "workshop": "azureml-h2o-customer",
        "model_format": manifest["model_format"],
        "model_h2o_version": manifest["h2o_version"],
        "runtime_h2o_version": manifest["runtime_h2o_version"],
        "mojo_version": manifest.get("mojo_version") or "not_applicable",
    },
)

print(
    f"Environment: "
    f"{environment_definition.name} (version assigned on registration)"
)
print(f"Base image: {environment_definition.image}")
display(environment_definition.tags)

## 4. Confirm the target workspace

Authentication uses your Azure CLI session. The next cell only reads the configured workspace and prints the registration switch. Check both values before allowing the final cell to write anything.

In [ ]:
credential = AzureCliCredential(
    tenant_id=os.getenv("AZURE_TENANT_ID") or None
)
ml_client = MLClient(
    credential,
    os.environ["AZURE_SUBSCRIPTION_ID"],
    os.environ["AZURE_RESOURCE_GROUP"],
    os.environ["AZUREML_WORKSPACE_NAME"],
)

workspace = ml_client.workspaces.get(os.environ["AZUREML_WORKSPACE_NAME"])
print(f"Target workspace: {workspace.name}")
print(f"Resource group: {os.environ['AZURE_RESOURCE_GROUP']}")
print(f"Environment registration enabled: {REGISTER}")

## 5. Register and verify

This is the only cell that writes to Azure. When registration is enabled, it creates the immutable environment version and reads it back to confirm that its H2O tag matches the customer manifest.

The first image build can take several minutes. After a successful run, return `REGISTER_H2O_ENVIRONMENT` to `false`.

In [ ]:
if REGISTER:
    registered_environment = ml_client.environments.create_or_update(
        environment_definition
,
)
    verified_environment = ml_client.environments.get(
        ENVIRONMENT_NAME,
        version=registered_environment.version,
    )

    assert (
        verified_environment.tags["runtime_h2o_version"]
        == manifest["runtime_h2o_version"]
    )
    print(
        f"Registered environment: "
        f"{verified_environment.name}:{verified_environment.version}"
    )
else:
    print(f"Prepared environment: {ENVIRONMENT_NAME} (version assigned on registration)")
    print(
        "Registration is off. Set REGISTER_H2O_ENVIRONMENT=true when you are ready."
    )

## Expected Result

The notebook-generated Conda definition pins the declared Python, Java, and H2O scoring runtime without changing the local `.venv`, and the immutable Azure ML environment is retrievable.

Next: `04_deploy_online_endpoint.ipynb`.